In [1]:
# ============================================================
# FUNCTION 3 - WEEK 13 FINAL-WEEK BAYESIAN OPTIMISATION
# Run from inside week13/
# ============================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)


# ------------------------------------------------------------
# 1. Load Week 13 cumulative data
# ------------------------------------------------------------

X = np.load(
    "function3/initial_inputs.npy"
)

Y = np.load(
    "function3/initial_outputs.npy"
).reshape(-1)

best_idx = np.argmax(Y)

best_x = X[best_idx]
best_y = Y[best_idx]

tree = cKDTree(X)

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(
    best_x,
    "->",
    best_y
)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 12 calibration check
# ------------------------------------------------------------

week12_x = np.array([
    0.383912,
    0.538429,
    0.432459
])

week12_actual = (
    -0.00352808609233225
)

# From Week 12 x1 profile
week12_pred_mean = (
    -0.010242377288633989
)

week12_pred_std = (
    0.009865669945765322
)

week12_error = (
    week12_actual
    - week12_pred_mean
)

week12_z_error = (
    week12_error
    / week12_pred_std
)

print("\n================================")
print("WEEK 12 CALIBRATION CHECK")
print("================================")

print(
    "Predicted mean:",
    week12_pred_mean
)

print(
    "Predicted std :",
    week12_pred_std
)

print(
    "Actual        :",
    week12_actual
)

print(
    "\nPrediction error:",
    week12_error
)

print(
    "\nError / predicted std:",
    week12_z_error
)


# ------------------------------------------------------------
# 3. Recent local evidence
# ------------------------------------------------------------

previous_best_x = np.array([
    0.332348,
    0.553987,
    0.420645
])

previous_best_y = (
    -0.002630184
)

week11_x = np.array([
    0.356112,
    0.538429,
    0.432459
])

week11_y = (
    -0.00126784051040226
)

print("\n================================")
print("RECENT LOCAL EVIDENCE")
print("================================")

print("\nPrevious strong point:")
print(
    previous_best_x,
    "->",
    previous_best_y
)

print("\nWeek 11 / incumbent:")
print(
    week11_x,
    "->",
    week11_y
)

print("\nWeek 12:")
print(
    week12_x,
    "->",
    week12_actual
)

print(
    "\nWeek 12 x1 displacement from incumbent:"
)

print(
    week12_x[0]
    - best_x[0]
)


# ------------------------------------------------------------
# 4. Fit updated ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(
            1e-3,
            1e3
        )
    )
    *
    Matern(
        length_scale=np.ones(3) * 0.2,
        length_scale_bounds=(
            0.01,
            2.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(
            1e-8,
            1e-1
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=15,
    random_state=42
)

gp.fit(
    X,
    Y
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(
    gp.kernel_
)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0 / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print(
    "\nARD lengthscales:"
)
print(
    lengthscales
)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(
    relative_sensitivity
)


# ------------------------------------------------------------
# 5. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu
        - best_y
        - xi
    )

    valid = (
        sigma > 1e-12
    )

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 6. Empirical local scale
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask]
    - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

empirical_cap = min(
    1.25 * nearest_distance,
    0.03
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print(
    "Nearest observed point:",
    nearest_distance
)

print(
    "\nEmpirical cap:",
    empirical_cap
)


# ------------------------------------------------------------
# 7. ARD-informed trust region
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.16 * lengthscales,
    0.012,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x
    - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x
    + trust_half_width
)

print("\n================================")
print("WEEK 13 TRUST REGION")
print("================================")

print("Centre:")
print(
    best_x
)

print(
    "\nHalf-widths:"
)
print(
    trust_half_width
)

print(
    "\nLower:"
)
print(
    lower
)

print(
    "\nUpper:"
)
print(
    upper
)


# ------------------------------------------------------------
# 8. Dense local candidate pool
# ------------------------------------------------------------

rng = np.random.default_rng(
    42
)

candidates = rng.uniform(
    lower,
    upper,
    size=(
        350000,
        3
    )
)

candidates = np.round(
    candidates,
    6
)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print(
    "\nCandidates after portal rounding + duplicate filtering:"
)

print(
    len(candidates)
)


# ------------------------------------------------------------
# 9. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 10. EI - diagnostic only
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(
    EI
)

print("\n================================")
print("PRIMARY EI - DIAGNOSTIC ONLY")
print("================================")

print(
    "candidate =",
    candidates[ei_idx]
)

print(
    "mean =",
    mu[ei_idx]
)

print(
    "std =",
    sigma[ei_idx]
)

print(
    "EI =",
    EI[ei_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 11. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(
    mu
)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "mean =",
    mu[mean_idx]
)

print(
    "std =",
    sigma[mean_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 12. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"\nbeta={beta}"
    )

    print(
        "candidate =",
        candidates[idx]
    )

    print(
        "mean =",
        mu[idx]
    )

    print(
        "std =",
        sigma[idx]
    )

    print(
        "distance =",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 13. Boundary check
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(
        len(x)
    ):

        if abs(
            x[j]
            - lower[j]
        ) <= tol:

            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(
            x[j]
            - upper[j]
        ) <= tol:

            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(
        status
    )


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )


# ------------------------------------------------------------
# 14. Axis profile diagnostics
# ------------------------------------------------------------

print("\n================================")
print("AXIS PROFILE DIAGNOSTICS")
print("================================")

for j in range(3):

    values = np.linspace(
        lower[j],
        upper[j],
        40000
    )

    profile = np.tile(
        best_x,
        (
            len(values),
            1
        )
    )

    profile[:, j] = values

    profile = np.round(
        profile,
        6
    )

    profile_distance, _ = tree.query(
        profile,
        k=1
    )

    profile_valid = profile[
        profile_distance > 0.01
    ]

    if len(profile_valid) == 0:

        print(
            f"\nx{j+1}: no admissible points"
        )

        continue

    profile_mu, profile_sigma = gp.predict(
        profile_valid,
        return_std=True
    )

    idx = np.argmax(
        profile_mu
    )

    print(
        f"\nx{j+1} profile best:"
    )

    print(
        "candidate =",
        profile_valid[idx]
    )

    print(
        "mean =",
        profile_mu[idx]
    )

    print(
        "std =",
        profile_sigma[idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            profile_valid[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 15. Week 12 mirror diagnostic
# ------------------------------------------------------------
#
# Week 12 moved x1 upward from the incumbent and got worse.
# Test the same displacement in the opposite direction.
# ------------------------------------------------------------

week12_step = (
    week12_x
    - best_x
)

mirror_candidate = (
    best_x
    - week12_step
)

mirror_candidate = np.clip(
    mirror_candidate,
    0.0,
    1.0
)

mirror_candidate = np.round(
    mirror_candidate,
    6
)

mirror_dist, mirror_idx = tree.query(
    mirror_candidate.reshape(1, -1),
    k=1
)

print("\n================================")
print("WEEK 12 MIRROR DIAGNOSTIC")
print("================================")

print(
    "Mirror candidate =",
    mirror_candidate
)

print(
    "Nearest observed distance =",
    mirror_dist[0]
)

print(
    "Nearest observed point =",
    X[mirror_idx[0]]
)

if mirror_dist[0] > 0.01:

    mirror_mu, mirror_sigma = gp.predict(
        mirror_candidate.reshape(1, -1),
        return_std=True
    )

    print(
        "mean =",
        mirror_mu[0]
    )

    print(
        "std =",
        mirror_sigma[0]
    )

else:

    print(
        "Mirror candidate is too close to an existing observation."
    )


# ------------------------------------------------------------
# 16. Previous-best -> current-best direction
# ------------------------------------------------------------
#
# This checks the empirical direction that produced the Week 11
# improvement and whether the updated GP prefers moving back
# toward the previous strong point or continuing beyond Week 11.
# ------------------------------------------------------------

direction = (
    best_x
    - previous_best_x
)

direction = (
    direction
    / np.linalg.norm(direction)
)

pos_limits = []
neg_limits = []

for j in range(3):

    if direction[j] > 0:

        pos_limits.append(
            (upper[j] - best_x[j])
            / direction[j]
        )

        neg_limits.append(
            (best_x[j] - lower[j])
            / direction[j]
        )

    elif direction[j] < 0:

        pos_limits.append(
            (lower[j] - best_x[j])
            / direction[j]
        )

        neg_limits.append(
            (best_x[j] - upper[j])
            / direction[j]
        )

t_pos_max = min(
    pos_limits
)

t_neg_max = min(
    neg_limits
)

t_values = np.linspace(
    -t_neg_max,
    t_pos_max,
    50000
)

line_candidates = (
    best_x
    + t_values[:, None]
    * direction
)

line_candidates = np.round(
    line_candidates,
    6
)

line_distance, _ = tree.query(
    line_candidates,
    k=1
)

line_valid = line_candidates[
    line_distance > 0.01
]

print("\n================================")
print("PREVIOUS-BEST -> INCUMBENT LINE")
print("================================")

if len(line_valid) > 0:

    line_mu, line_sigma = gp.predict(
        line_valid,
        return_std=True
    )

    line_idx = np.argmax(
        line_mu
    )

    print(
        "Best line candidate =",
        line_valid[line_idx]
    )

    print(
        "mean =",
        line_mu[line_idx]
    )

    print(
        "std =",
        line_sigma[line_idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            line_valid[line_idx]
            - best_x
        )
    )

else:

    print(
        "No admissible line candidates."
    )

DATA
X shape: (27, 3)
Y shape: (27,)

Current best:
[0.356112 0.538429 0.432459] -> -0.00126784051040226

Y range:
min = -0.3989255131463011
max = -0.00126784051040226
std = 0.07595693937050914

WEEK 12 CALIBRATION CHECK
Predicted mean: -0.010242377288633989
Predicted std : 0.009865669945765322
Actual        : -0.00352808609233225

Prediction error: 0.006714291196301739

Error / predicted std: 0.6805712367444178

RECENT LOCAL EVIDENCE

Previous strong point:
[0.332348 0.553987 0.420645] -> -0.002630184

Week 11 / incumbent:
[0.356112 0.538429 0.432459] -> -0.00126784051040226

Week 12:
[0.383912 0.538429 0.432459] -> -0.00352808609233225

Week 12 x1 displacement from incumbent:
0.02779999999999999

GP FIT

Fitted kernel:
1.84**2 * Matern(length_scale=[0.809, 1.24, 0.286], nu=2.5) + WhiteKernel(noise_level=0.012)

ARD lengthscales:
[0.80854307 1.240926   0.28595712]

Normalised inverse-lengthscale sensitivity:
[0.22326104 0.14546892 0.63127005]

EMPIRICAL LOCAL SCALE
Nearest observed po

In [2]:
# ============================================================
# FINAL FUNCTION 3 - WEEK 13 SELECTION
# ============================================================
#
# FINAL CAPSTONE QUERY
#
# Evidence:
# - Week 11 produced the current best.
# - Week 12 increased x1 alone and performed worse, suggesting
#   that the one-coordinate continuation overshot the useful
#   local region.
# - The updated GP still prefers higher x1, but the axis
#   profiles simultaneously favour:
#       x1 higher
#       x2 lower
#       x3 higher
# - The previous-best -> incumbent direction combines exactly
#   these three movements.
# - Its predicted mean is very close to the unrestricted
#   highest-mean candidate, but it avoids pushing x1 as far
#   toward the unsuccessful Week 12 location.
#
# For the final query, choose the more structured and
# conservative coordinated refinement.
# ============================================================

week13_candidate = np.array([
    0.376500,
    0.525082,
    0.442594
])

print("================================")
print("FUNCTION 3 - WEEK 13 FINAL")
print("================================")

print("\nCandidate:")
print(week13_candidate)


# ------------------------------------------------------------
# GP prediction
# ------------------------------------------------------------

final_mu, final_sigma = gp.predict(
    week13_candidate.reshape(1, -1),
    return_std=True
)

print("\nPredicted mean:")
print(final_mu[0])

print("\nPredicted std:")
print(final_sigma[0])


# ------------------------------------------------------------
# Distance from incumbent
# ------------------------------------------------------------

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week13_candidate - best_x
    )
)


# ------------------------------------------------------------
# Check against all previous observations
# ------------------------------------------------------------

nearest_dist, nearest_idx = tree.query(
    week13_candidate.reshape(1, -1),
    k=1
)

print("\nNearest observed-point distance:")
print(nearest_dist[0])

print("\nNearest observed point:")
print(X[nearest_idx[0]])

print("\nDuplicate rule satisfied:")
print(
    nearest_dist[0] > 0.01
)


# ------------------------------------------------------------
# Portal format
# ------------------------------------------------------------

portal = "-".join(
    f"{x:.6f}"
    for x in week13_candidate
)

print("\nPortal format:")
print(portal)

FUNCTION 3 - WEEK 13 FINAL

Candidate:
[0.3765   0.525082 0.442594]

Predicted mean:
-0.00882239542980115

Predicted std:
0.00921039254175269

Distance from current best:
0.026391877121569065

Nearest observed-point distance:
0.018324802263598908

Nearest observed point:
[0.383912 0.538429 0.432459]

Duplicate rule satisfied:
True

Portal format:
0.376500-0.525082-0.442594
